# 🧠 Lesson 4.1 — The Artificial Neuron

**AI Crash Course for Absolute Beginners**

Every neural network is built from one simple unit: the artificial neuron. In this notebook:
- Build a neuron from scratch in Python
- Understand weighted inputs, bias, and activation functions
- Implement a Perceptron and watch it learn
- Explore sigmoid, ReLU, and tanh activations

> Run each cell with **Shift + Enter**.

In [ ]:
# Install libraries (already in Colab — this is just a safety net)
!pip install numpy matplotlib --quiet

# numpy — fast maths and array operations (used in almost every AI project)
# "as np" is a shortcut so we can write np.array() instead of numpy.array()
import numpy as np

# matplotlib — for drawing charts and graphs
# "as plt" is a shortcut so we can write plt.plot() instead of matplotlib.pyplot.plot()
import matplotlib.pyplot as plt

---
## Part 1 — A Neuron is Just Weighted Addition + Activation

In [ ]:
# A neuron receives inputs, multiplies by weights, adds bias, then squashes
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def neuron(inputs, weights, bias, activation=sigmoid):
    """One artificial neuron."""
    z = np.dot(inputs, weights) + bias   # weighted sum
    return activation(z)                 # squash to output

# Example: predict if someone will click an ad
# inputs: [time_on_page(minutes), age, prev_purchases]
inputs  = np.array([3.5, 28, 2])
weights = np.array([0.8, -0.2, 1.1])     # learned during training
bias    = -1.5

raw_sum = np.dot(inputs, weights) + bias
output  = sigmoid(raw_sum)

print(f"Weighted sum (z) : {raw_sum:.3f}")
print(f"Sigmoid output   : {output:.3f}")
print(f"Prediction       : {'Will click' if output > 0.5 else 'Will not click'} ({output:.1%} probability)")

---
## Part 2 — Activation Functions: The Neuron's On/Off Switch

In [ ]:
def relu(z):   return np.maximum(0, z)
def tanh(z):   return np.tanh(z)

z = np.linspace(-5, 5, 300)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

funcs = [
    (sigmoid, "Sigmoid",  "Output: 0 to 1 (probabilities)",    "#c8401a"),
    (tanh,    "Tanh",     "Output: -1 to 1 (zero-centred)",    "#1a6bc8"),
    (relu,    "ReLU",     "Output: 0 or z (fast, sparse)",     "#2a8c5a"),
]

for ax, (fn, name, note, color) in zip(axes, funcs):
    ax.plot(z, fn(z), color=color, linewidth=2.5)
    ax.axhline(0, color="grey", linewidth=0.5)
    ax.axvline(0, color="grey", linewidth=0.5)
    ax.set_title(f"{name}\n{note}", fontsize=9)
    ax.set_xlabel("z")
    ax.set_ylabel("activation(z)")

plt.suptitle("Activation Functions", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# When to use which?
print("Which activation to use?")
print("  Hidden layers  -> ReLU (default), Leaky ReLU, GELU")
print("  Output (binary classification) -> Sigmoid")
print("  Output (multi-class)           -> Softmax")
print("  Output (regression)            -> None (linear)")

---
## Part 3 — The Perceptron: The Simplest Neural Network

In [ ]:
class Perceptron:
    """Single-layer linear classifier (Rosenblatt 1957).
    The simplest possible neural network — just one neuron with a step activation.
    """

    def __init__(self, n_features, lr=0.1):
        # Start weights at zero — one weight per input feature
        self.weights = np.zeros(n_features)
        self.bias    = 0.0    # bias shifts the decision boundary
        self.lr      = lr     # learning rate: how big a step to take when correcting mistakes

    def predict(self, X):
        # Calculate weighted sum + bias, then apply step function:
        # if result >= 0 → predict class 1, otherwise → predict class 0
        # astype(int) converts True/False to 1/0
        return (np.dot(X, self.weights) + self.bias >= 0).astype(int)

    def fit(self, X, y, epochs=20):
        history = []   # track number of errors each epoch
        for epoch in range(epochs):
            errors = 0
            # zip(X, y) pairs each input row with its correct label
            for xi, yi in zip(X, y):
                pred  = self.predict(xi)
                error = yi - pred          # 0 = correct, +1 = should be 1 but predicted 0, -1 = opposite
                # Perceptron learning rule: nudge weights in the direction that reduces the error
                self.weights += self.lr * error * xi
                self.bias    += self.lr * error
                errors += int(error != 0)  # count this as a mistake if error is not zero
            history.append(errors)
            if errors == 0:               # stop early if perfectly classified
                break
        return history


# Learn the AND gate
# AND is true only when BOTH inputs are 1
X_and = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=float)
y_and = np.array([0,0,0,1])   # only [1,1] gives output 1

p = Perceptron(n_features=2, lr=0.1)
errors = p.fit(X_and, y_and, epochs=50)

print("AND Gate predictions:")
for xi, yi in zip(X_and, y_and):
    pred = p.predict(xi)
    print(f"  {int(xi[0])} AND {int(xi[1])} = {pred}  (true: {yi})  {'OK' if pred==yi else 'WRONG'}")

# Plot how the number of mistakes decreases over training epochs
plt.figure(figsize=(5, 3))
plt.plot(errors, marker="o", color="#c8401a")
plt.xlabel("Epoch")
plt.ylabel("Misclassifications")
plt.title("Perceptron Learning: AND Gate")
plt.tight_layout()
plt.show()

---
## Part 4 — The XOR Problem: Why We Need Layers

In [ ]:
X_xor = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=float)
y_xor = np.array([0,1,1,0])   # XOR is NOT linearly separable

p2 = Perceptron(n_features=2, lr=0.1)
errors_xor = p2.fit(X_xor, y_xor, epochs=100)

print("XOR Gate predictions:")
for xi, yi in zip(X_xor, y_xor):
    pred = p2.predict(xi)
    print(f"  {int(xi[0])} XOR {int(xi[1])} = {pred}  (true: {yi})  {'OK' if pred==yi else 'WRONG'}")

print("\nThe Perceptron CANNOT learn XOR — we need a multi-layer network.")
print("That's exactly what Lesson 4.2 covers!")

---
## Part 5 — A Single Sigmoid Neuron for Binary Classification

In [ ]:
class SigmoidNeuron:
    """A single neuron with sigmoid activation trained using gradient descent.
    This is mathematically identical to logistic regression.
    """

    def __init__(self, n_features, lr=0.1):
        self.w  = np.zeros(n_features)  # weights, one per feature
        self.b  = 0.0                   # bias term
        self.lr = lr                    # learning rate

    def _sigmoid(self, z):
        # Squashes any number into the range 0 to 1 (used for probabilities)
        return 1 / (1 + np.exp(-z))

    def forward(self, X):
        # X @ self.w is matrix multiplication (same as np.dot)
        # This computes the weighted sum for every sample at once
        return self._sigmoid(X @ self.w + self.b)

    def fit(self, X, y, epochs=200):
        losses = []
        for _ in range(epochs):
            y_hat = self.forward(X)     # get current predictions (probabilities)

            # Cross-entropy loss: measures how wrong the predictions are
            # 1e-9 is a tiny number added to avoid log(0) which would cause an error
            loss = -np.mean(y * np.log(y_hat + 1e-9) + (1 - y) * np.log(1 - y_hat + 1e-9))

            # Calculate gradients (how much to adjust each weight)
            err = y_hat - y             # prediction error for each sample

            # X.T is the transpose of X (flips rows and columns)
            # Dividing by len(y) averages the gradient across all samples
            self.w -= self.lr * (X.T @ err) / len(y)
            self.b -= self.lr * err.mean()

            losses.append(loss)
        return losses

# --- Create a simple dataset ---
# np.random.seed(0) makes the random numbers the same every time you run this cell
np.random.seed(0)

# Two clusters of points — class 1 centred at (2,2), class 0 centred at (-2,-2)
X_pos = np.random.randn(50, 2) + [2, 2]    # 50 samples for class 1
X_neg = np.random.randn(50, 2) + [-2, -2]  # 50 samples for class 0

# np.vstack stacks the two arrays on top of each other (like stacking two tables of rows)
X_data = np.vstack([X_pos, X_neg])
y_data = np.array([1]*50 + [0]*50, dtype=float)  # labels: 50 ones then 50 zeros

# --- Train ---
sn = SigmoidNeuron(2, lr=0.5)
losses = sn.fit(X_data, y_data, epochs=300)

plt.figure(figsize=(12, 4))

# --- Plot 1: loss going down over time ---
plt.subplot(1, 2, 1)
plt.plot(losses, color="#c8401a")
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy Loss")
plt.title("Training Loss (should go down)")

# --- Plot 2: decision boundary ---
plt.subplot(1, 2, 2)
plt.scatter(X_pos[:,0], X_pos[:,1], color="#1a6bc8", label="Class 1", alpha=0.5)
plt.scatter(X_neg[:,0], X_neg[:,1], color="#c8401a", label="Class 0", alpha=0.5)

# Draw the decision boundary line where the neuron is exactly 50/50 uncertain
# Equation: w0*x + w1*y + b = 0  →  y = -(w0*x + b) / w1
x_line = np.linspace(-5, 6, 100)
y_line = -(sn.w[0] * x_line + sn.b) / sn.w[1]
plt.plot(x_line, y_line, "k--", label="Decision boundary")
plt.legend()
plt.title("Sigmoid Neuron Decision Boundary")

plt.tight_layout()
plt.show()

# Accuracy: predict class 1 if probability > 0.5, otherwise class 0
preds = (sn.forward(X_data) > 0.5).astype(int)
print(f"Training accuracy: {(preds == y_data).mean():.1%}")

---
## Challenge Exercises

1. **OR gate**: Adapt the Perceptron to learn the OR gate. Does it converge faster than AND?
2. **Leaky ReLU**: Implement `leaky_relu(z, alpha=0.01)` — what happens for negative z?
3. **Softmax**: Implement `softmax(z) = exp(z) / sum(exp(z))` for multi-class output. Note how all outputs sum to 1.

---
**Next lesson:** [Lesson 4.2 — Building a Neural Network](https://colab.research.google.com/github/GirlEf/ai-crash-course/blob/main/notebooks/lesson-4.2-neural-network.ipynb)